# Modele sklearn

## Wstępna konfiguracja

### Importowanie bibliotek

In [1]:
import pandas as pd

from mlxtend.data import loadlocal_mnist

from sklearn import model_selection
from sklearn import metrics

from sklearn import linear_model
from sklearn import tree
from sklearn import ensemble
from sklearn import neural_network

### Inicjalizacja konfiguracji

In [2]:
class config:
    
    # Ścieżki do plików
    DOWNLOADED_IMAGES_PATH = "data/t10k-images.idx3-ubyte"
    DOWNLOADED_LABELS_PATH = "data/t10k-labels.idx1-ubyte"

    # Ogólne ustawienia projektu
    RANDOMIZE_DATA = True
    FOLDS_CNT = 5

## Konfiguracja danych

### Wczytanie danych

In [3]:
# Wczytanie danych z plików
X, y = loadlocal_mnist(
    images_path=config.DOWNLOADED_IMAGES_PATH,
    labels_path=config.DOWNLOADED_LABELS_PATH
)

# Generowanie kolumn z id pikseli
pixel_columns = [f"pixel{i}" for i in range(len(X[0]))]

# Stworzenie pandas DataFrame
df = pd.DataFrame(X, columns=pixel_columns)

# Dodanie etykiety
df["label"] = y

# Wyświetlenie danych
df

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,7
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
9996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
9997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
9998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5


### Podział danych na foldy

<img src="./images/image1.png" alt="image1" width="1300"/>
<!-- ![image1](./images/image1.png) -->
<!-- ![image1](https://towardsdatascience.com/wp-content/uploads/2023/12/1N45hocCMP0u4nXLe0WuSvw.png) -->

In [4]:
# Stworzenie kolumny kfold
df["kfold"] = -1

# Podział danych na segmenty oraz ewentualnie przelosowanie danych
folds_model = model_selection.StratifiedKFold(
    n_splits=config.FOLDS_CNT,
    shuffle=config.RANDOMIZE_DATA
)
for fold, (train, test) in enumerate(folds_model.split(df, df["label"].values)):
    df.loc[test, "kfold"] = fold
    
    print(f"{fold}. {train}, {test}")

# Wyświetlenie danych
df

0. [   0    1    3 ... 9997 9998 9999], [   2    5    9 ... 9972 9977 9990]
1. [   1    2    3 ... 9994 9995 9997], [   0    6   10 ... 9996 9998 9999]
2. [   0    1    2 ... 9997 9998 9999], [  17   28   31 ... 9992 9993 9995]
3. [   0    2    3 ... 9997 9998 9999], [   1    4    7 ... 9979 9983 9989]
4. [   0    1    2 ... 9996 9998 9999], [   3    8   13 ... 9991 9994 9997]


,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,label,kfold
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,7,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,3
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,2
9996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,1
9997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,4
9998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,5,1


## Trenowanie modelu

### Dyspozytor modelu

In [5]:
class model_dispatcher:
    
    # Tablica z nazwami modeli
    model_names = [
        # "logistic_regression",
        "SGDClassifier",
        "decision_tree_gini",
        "decision_tree_entropy",
        "decision_tree_log_loss",
        "random_forest",
        "neural_network"
    ]

    # Słownik z nazwami modeli oraz klasami
    models = {
        "logistic_regression": linear_model.LogisticRegression(
            max_iter=6000
        ),
        "SGDClassifier": linear_model.SGDClassifier(),
        "decision_tree_gini": tree.DecisionTreeClassifier(
            criterion="gini"
        ),
        "decision_tree_entropy": tree.DecisionTreeClassifier(
            criterion="entropy"
        ),
        "decision_tree_log_loss": tree.DecisionTreeClassifier(
            criterion="log_loss"
        ),
        "random_forest": ensemble.RandomForestClassifier(),
        "neural_network": neural_network.MLPClassifier()
    }

### Trenowanie oraz testowanie modelu

In [9]:
def use_model(df_train, df_test, model, metrics_func):
    
    # Podział danych na odpowiednie zmienne
    x_train = df_train.loc[:, df_train.columns != "label"].values
    y_train = df_train.loc[:, "label"].values

    x_test = df_test.loc[:, df_test.columns != "label"].values
    y_test = df_test.loc[:, "label"].values
    
    # Trenowanie modelu
    model.fit(x_train, y_train)

    # Wykorzystanie modelu
    predicted_data = model.predict(x_test)
    
    # Obliczenie oraz zwrócenie wyników metryki
    scores = metrics_func(
        y_true=y_test,
        y_pred=predicted_data
    )
    return scores

In [10]:
# Inicjalizacja słownika z modelami oraz metrykami ich wyników
metrics_dict = {}

# Pętla przechodząca przez modele
for model_name in model_dispatcher.model_names:
    model = model_dispatcher.models[model_name]
    
    # Inicjalizacja nowego elementu w słowniku
    metrics_dict[model_name] = []

    # Testowanie modelu na różnych foldach
    for fold in range(config.FOLDS_CNT):
        
        # Inicjalizacja train oraz test dataframe
        df_train = df.loc[df["kfold"] != fold, df.columns != "kfold"]
        df_test = df.loc[df["kfold"] == fold, df.columns != "kfold"]

        # Obliczanie accuracy
        score = use_model(df_train, df_test, model, metrics.accuracy_score)

        # Dodanie accuracy do tablicy wyników aktualnego modelu
        metrics_dict[model_name].append(score)

        # Wypisanie accuracy
        print(f"{model_name}: {fold} - {score:.3f}")

SGDClassifier: 0 - 0.864
SGDClassifier: 1 - 0.876
SGDClassifier: 2 - 0.869
SGDClassifier: 3 - 0.882
SGDClassifier: 4 - 0.867
decision_tree_gini: 0 - 0.797
decision_tree_gini: 1 - 0.821
decision_tree_gini: 2 - 0.807
decision_tree_gini: 3 - 0.806
decision_tree_gini: 4 - 0.790
decision_tree_entropy: 0 - 0.813
decision_tree_entropy: 1 - 0.823
decision_tree_entropy: 2 - 0.803
decision_tree_entropy: 3 - 0.810
decision_tree_entropy: 4 - 0.807
decision_tree_log_loss: 0 - 0.806
decision_tree_log_loss: 1 - 0.816
decision_tree_log_loss: 2 - 0.811
decision_tree_log_loss: 3 - 0.809
decision_tree_log_loss: 4 - 0.811
random_forest: 0 - 0.958
random_forest: 1 - 0.959
random_forest: 2 - 0.945
random_forest: 3 - 0.948
random_forest: 4 - 0.951
neural_network: 0 - 0.920
neural_network: 1 - 0.932
neural_network: 2 - 0.919
neural_network: 3 - 0.924
neural_network: 4 - 0.888


## Metryki

### Słownik metryk

In [8]:
metrics_dict

{'SGDClassifier': [0.884, 0.8825, 0.8685, 0.8945, 0.8645],
 'decision_tree_gini': [0.814, 0.7955, 0.805, 0.8205, 0.802],
 'decision_tree_entropy': [0.822, 0.8195, 0.8175, 0.829, 0.8075],
 'decision_tree_log_loss': [0.8175, 0.821, 0.81, 0.827, 0.807],
 'random_forest': [0.9595, 0.9515, 0.949, 0.9555, 0.9415],
 'neural_network': [0.916, 0.9095, 0.928, 0.914, 0.9095]}

### Metryki w DataFrame-ach

In [9]:
metrics_df = pd.DataFrame(metrics_dict)

display(metrics_df)
display(metrics_df.mean().to_frame().transpose().rename(index={0: "avg"}))

,SGDClassifier,decision_tree_gini,decision_tree_entropy,decision_tree_log_loss,random_forest,neural_network
0,0.8840,0.8140,0.8220,0.8175,0.9595,0.9160
1,0.8825,0.7955,0.8195,0.8210,0.9515,0.9095
2,0.8685,0.8050,0.8175,0.8100,0.9490,0.9280
3,0.8945,0.8205,0.8290,0.8270,0.9555,0.9140
4,0.8645,0.8020,0.8075,0.8070,0.9415,0.9095


,SGDClassifier,decision_tree_gini,decision_tree_entropy,decision_tree_log_loss,random_forest,neural_network
avg,0.8788,0.8074,0.8191,0.8165,0.9514,0.9154


In [11]:
metrics_df_tran = pd.DataFrame(metrics_dict).transpose()

display(metrics_df_tran)
display(metrics_df_tran.mean(axis=1).to_frame().rename(columns={0: "avg"}))

,0,1,2,3,4
SGDClassifier,0.8840,0.8825,0.8685,0.8945,0.8645
decision_tree_gini,0.8140,0.7955,0.8050,0.8205,0.8020
decision_tree_entropy,0.8220,0.8195,0.8175,0.8290,0.8075
decision_tree_log_loss,0.8175,0.8210,0.8100,0.8270,0.8070
random_forest,0.9595,0.9515,0.9490,0.9555,0.9415
neural_network,0.9160,0.9095,0.9280,0.9140,0.9095


,avg
SGDClassifier,0.8788
decision_tree_gini,0.8074
decision_tree_entropy,0.8191
decision_tree_log_loss,0.8165
random_forest,0.9514
neural_network,0.9154
